# 🏆 Forecasting the Beautiful Game
## Bayesian Inference & Graphical Models for World Cup 2026 Prediction
**LASR 2026 Workshop · University of Leeds**

This notebook fits a **Dixon-Coles model** on 49,000+ international football results, simulates the 2026 World Cup 50,000 times, and produces all six visualisations used in the conference poster.

| Section | Content |
|---------|---------|
| A | Install dependencies & imports |
| B | Load & clean data |
| C | Fit Dixon-Coles model |
| D | Simulate tournament |
| E | Visualisations |
| F | Optional: PyMC full Bayesian model |

**Dataset:** Auto-downloaded from [martj42/international_results](https://github.com/martj42/international_results) — no manual download needed.

## A · Install Dependencies & Imports

In [ ]:
# Run this cell first — installs everything needed
!pip install scipy pandas numpy matplotlib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import poisson, gamma as sp_gamma
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

# ── Colour palette (matches poster) ──────────────────────────
NAVY  = "#0A1628"
TEAL  = "#0D9488"
GOLD  = "#F59E0B"
LIGHT = "#E0F2F1"
GRAY  = "#64748B"
WHITE = "#FFFFFF"

plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
})

print("✅  Imports done")

## B · Load & Clean Data

**Source:** `martj42/international_results` on GitHub
- 49,000+ results from 1872 to present
- Free, public domain, no login required
- Columns: `date`, `home_team`, `away_team`, `home_score`, `away_score`, `tournament`, `city`, `country`, `neutral`

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "martj42/international_results/master/results.csv"
)

print("Downloading data …")
df_raw = pd.read_csv(DATA_URL)
print(f"  Loaded {len(df_raw):,} rows")
df_raw.head()

In [ ]:
# ── Filter: 2010-present, major tournaments + friendlies ─────
df = df_raw.copy()
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].copy()
df = df[df["tournament"].isin([
    "FIFA World Cup", "FIFA World Cup qualification",
    "UEFA Euro", "UEFA Euro qualification",
    "Copa América", "AFC Asian Cup",
    "Africa Cup of Nations", "Friendly"
])].copy()
df = df.dropna(subset=["home_score", "away_score"])
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)
print(f"After filter: {len(df):,} matches (2010-present)")

# ── Keep only WC 2026 contenders ─────────────────────────────
WC_TEAMS = [
    "France", "Brazil", "England", "Spain", "Argentina", "Germany",
    "Portugal", "Netherlands", "Belgium", "Croatia", "Denmark",
    "Uruguay", "Colombia", "Mexico", "United States",
    "Japan", "Morocco", "Senegal", "Australia", "South Korea",
]

df_wc = df[
    df["home_team"].isin(WC_TEAMS) & df["away_team"].isin(WC_TEAMS)
].copy().reset_index(drop=True)

print(f"WC contender matches: {len(df_wc):,}")
print(f"Teams: {len(WC_TEAMS)}")
df_wc.head(10)

In [ ]:
# ── Quick look at goal distributions ─────────────────────────
print("Home goals distribution:")
print(df_wc["home_score"].value_counts().sort_index().head(8).to_string())
print()
print("Away goals distribution:")
print(df_wc["away_score"].value_counts().sort_index().head(8).to_string())
print()
print(f"Mean home goals: {df_wc['home_score'].mean():.3f}")
print(f"Mean away goals: {df_wc['away_score'].mean():.3f}")
print(f"Home win rate:   {(df_wc['home_score'] > df_wc['away_score']).mean():.1%}")

## C · Fit the Dixon-Coles Model

The **Dixon-Coles (1997)** model assumes goals are Poisson-distributed:

$$G_{ij} \sim \text{Poisson}(\lambda_{ij}), \quad \lambda_{ij} = \alpha_i \cdot \delta_j \cdot \gamma$$

where $\alpha_i$ = attack strength, $\delta_j$ = defence strength, $\gamma$ = home advantage.

A correction term $\tau_{\rho}$ is added for low-scoring games (0-0, 1-0, 0-1, 1-1) where the Poisson independence assumption breaks down.

Parameters are estimated by **maximum likelihood** with bounded SLSQP optimisation.

In [ ]:
teams    = sorted(set(df_wc["home_team"]) | set(df_wc["away_team"]))
team_idx = {t: i for i, t in enumerate(teams)}
N        = len(teams)
print(f"Teams in model: {N}")
print(teams)

In [ ]:
# ── Dixon-Coles low-score correction ─────────────────────────
def tau(x, y, lam, mu, rho):
    """Correction factor for low-scoring games."""
    if   x == 0 and y == 0: return max(1e-6, 1 - lam * mu * rho)
    elif x == 1 and y == 0: return max(1e-6, 1 + mu  * rho)
    elif x == 0 and y == 1: return max(1e-6, 1 + lam * rho)
    elif x == 1 and y == 1: return max(1e-6, 1 - rho)
    else:                    return 1.0

# ── Negative log-likelihood ───────────────────────────────────
def neg_log_lik(params):
    att = params[:N]
    dfc = params[N:2*N]
    ha  = params[2*N]
    rho = params[2*N + 1]
    ll  = 0.0
    for _, row in df_wc.iterrows():
        i  = team_idx[row["home_team"]]
        j  = team_idx[row["away_team"]]
        x  = int(row["home_score"])
        y  = int(row["away_score"])
        lm = np.exp(att[i] - dfc[j] + ha)
        mu = np.exp(att[j] - dfc[i])
        ll += (np.log(tau(x, y, lm, mu, rho))
               + poisson.logpmf(x, lm)
               + poisson.logpmf(y, mu))
    return -ll

# ── Optimise ──────────────────────────────────────────────────
x0 = np.zeros(2 * N + 2)
x0[2*N]     =  0.15    # home advantage start
x0[2*N + 1] = -0.05    # rho start

bounds = (
    [(-3, 3)] * N  +   # attack bounds
    [(-3, 3)] * N  +   # defence bounds
    [(0.0, 0.5)]   +   # home advantage
    [(-0.5, 0.1)]      # rho
)
constraints = [{"type": "eq", "fun": lambda p: p[:N].sum()}]  # sum(attack) = 0

print("Fitting Dixon-Coles model (may take ~30s) …")
res = minimize(
    neg_log_lik, x0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"maxiter": 1000, "ftol": 1e-9}
)

p      = res.x
att_p  = {t: p[team_idx[t]]      for t in teams}
def_p  = {t: p[N + team_idx[t]]  for t in teams}
ha     = p[2*N]
rho    = p[2*N + 1]

print(f"\nOptimisation success : {res.success}")
print(f"Home advantage (log) : {ha:.4f}  →  multiplier {np.exp(ha):.3f}x")
print(f"Rho (DC correction)  : {rho:.4f}")

In [ ]:
# ── Print ranked team strengths ───────────────────────────────
print("── Attack rankings ──────────────────────────────────────")
for rank, (t, v) in enumerate(
        sorted(att_p.items(), key=lambda x: x[1], reverse=True), 1):
    bar = "█" * int((v + 2) * 8)
    print(f"  {rank:2}. {t:<20} {v:+.3f}  {bar}")

print()
print("── Defence rankings (lower = stronger) ──────────────────")
for rank, (t, v) in enumerate(
        sorted(def_p.items(), key=lambda x: x[1]), 1):
    bar = "█" * max(0, int((2 - v) * 8))
    print(f"  {rank:2}. {t:<20} {v:+.3f}  {bar}")

In [ ]:
# ── Helper: predicted Poisson rates for any match ────────────
def match_rates(home, away, neutral=False):
    """Return (lambda_home, lambda_away) Poisson goal rates."""
    lm = np.exp(att_p[home] - def_p[away] + (0 if neutral else ha))
    mu = np.exp(att_p[away] - def_p[home])
    return lm, mu

def sim_match(home, away, neutral=False):
    """Simulate a single match; return (home_goals, away_goals)."""
    lm, mu = match_rates(home, away, neutral)
    return np.random.poisson(lm), np.random.poisson(mu)

# ── Sanity check: France vs England ──────────────────────────
lm, mu = match_rates("France", "England", neutral=True)
print(f"France vs England (neutral):")
print(f"  France expected goals  : {lm:.3f}")
print(f"  England expected goals : {mu:.3f}")
p_fr = sum(poisson.pmf(hg,lm)*sum(poisson.pmf(ag,mu) for ag in range(hg)) for hg in range(1,9))
p_en = sum(poisson.pmf(ag,mu)*sum(poisson.pmf(hg,lm) for hg in range(ag)) for ag in range(1,9))
print(f"  P(France win)          : {p_fr:.1%}")
print(f"  P(England win)         : {p_en:.1%}")
print(f"  P(Draw)                : {1-p_fr-p_en:.1%}")

## D · Tournament Simulation

We simulate a knockout bracket with the 20 WC teams **50,000 times**, sampling goals from the fitted Poisson distributions at each stage. Draws go to 50/50 penalties.

In [ ]:
def simulate_tournament(team_list, n=50_000):
    """Simulate a knockout tournament n times. Returns win % per team."""
    wins  = {t: 0 for t in team_list}
    pool0 = list(team_list)
    for _ in range(n):
        pool = pool0[:]
        np.random.shuffle(pool)
        while len(pool) > 1:
            nxt = []
            for k in range(0, len(pool), 2):
                if k + 1 >= len(pool):
                    nxt.append(pool[k]); continue
                h, a   = pool[k], pool[k+1]
                hg, ag = sim_match(h, a, neutral=True)
                if   hg > ag: nxt.append(h)
                elif ag > hg: nxt.append(a)
                else:         nxt.append(h if np.random.random() < .5 else a)
            pool = nxt
        wins[pool[0]] += 1
    return {t: wins[t] / n * 100 for t in team_list}

print("Simulating 50,000 tournaments …")
win_p  = simulate_tournament(WC_TEAMS, n=50_000)
win_ps = dict(sorted(win_p.items(), key=lambda x: x[1], reverse=True))

print("\nFull win probability table:")
print(f"{'Rank':<5} {'Team':<22} {'Win %':>7}")
print("-" * 38)
for rank, (t, p) in enumerate(win_ps.items(), 1):
    bar = "▓" * int(p * 2)
    print(f"  {rank:<4} {t:<22} {p:>6.2f}%  {bar}")

## E · Visualisations

All six charts used in the LASR 2026 poster.

### E1 · Poisson Goal Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
lm_fr, lm_en = match_rates("France", "England", neutral=True)
k  = np.arange(0, 8)
w  = 0.38
ax.bar(k - w/2, poisson.pmf(k, lm_fr), w,
       color=TEAL, alpha=0.9, label=f"France   λ = {lm_fr:.2f}", zorder=3)
ax.bar(k + w/2, poisson.pmf(k, lm_en), w,
       color=GOLD, alpha=0.9, label=f"England  λ = {lm_en:.2f}", zorder=3)
ax.set_xlabel("Goals scored", color=NAVY, fontweight="bold", fontsize=11)
ax.set_ylabel("P(X = k)",     color=NAVY, fontweight="bold", fontsize=11)
ax.set_title("Poisson Goal Model – France vs England\n(fitted Dixon-Coles parameters)",
             color=NAVY, fontweight="bold", fontsize=12)
ax.legend(framealpha=0, labelcolor=NAVY, fontsize=10)
ax.yaxis.grid(True, color="#CBD5E1", lw=0.6, zorder=0)
ax.set_xticks(k); ax.tick_params(colors=NAVY)
plt.tight_layout()
plt.savefig("viz_1_poisson.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved → viz_1_poisson.png")

### E2 · Team Attack & Defence Parameters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
med_att = np.median([att_p[t] for t in WC_TEAMS])
med_def = np.median([def_p[t] for t in WC_TEAMS])

wc_att = sorted(WC_TEAMS, key=lambda t: att_p[t])
ax = axes[0]
ax.barh(wc_att,
        [att_p[t] for t in wc_att],
        color=[TEAL if att_p[t] > med_att else GRAY for t in wc_att],
        alpha=0.9)
ax.axvline(0, color=NAVY, lw=0.8, ls="--")
ax.set_title("Attack Strength  (αᵢ)", color=NAVY, fontweight="bold", fontsize=11)
ax.set_xlabel("Log attack parameter", color=NAVY)
ax.tick_params(colors=NAVY, labelsize=9)

wc_def = sorted(WC_TEAMS, key=lambda t: def_p[t])
ax = axes[1]
ax.barh(wc_def,
        [def_p[t] for t in wc_def],
        color=[GOLD if def_p[t] < med_def else GRAY for t in wc_def],
        alpha=0.9)
ax.axvline(0, color=NAVY, lw=0.8, ls="--")
ax.set_title("Defence Strength  (δⱼ) — lower = stronger",
             color=NAVY, fontweight="bold", fontsize=11)
ax.set_xlabel("Log defence parameter", color=NAVY)
ax.tick_params(colors=NAVY, labelsize=9)

fig.suptitle("Dixon-Coles Team Parameters – WC 2026 Contenders",
             color=NAVY, fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("viz_2_team_params.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved → viz_2_team_params.png")

### E3 · Tournament Win Probabilities

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
t_list = list(reversed(list(win_ps.keys())))
p_list = [win_ps[t] for t in t_list]
thresh = sorted(p_list)[-5]
cols   = [TEAL if p >= thresh else GRAY for p in p_list]

bars = ax.barh(t_list, p_list, color=cols, alpha=0.9, height=0.65)
for bar, p in zip(bars, p_list):
    ax.text(bar.get_width() + 0.1,
            bar.get_y() + bar.get_height() / 2,
            f"{p:.1f}%", va="center", color=NAVY, fontsize=9.5, fontweight="bold")

ax.set_xlabel("Win probability (%)", color=NAVY, fontweight="bold", fontsize=11)
ax.set_title("2026 FIFA World Cup – Win Probabilities\n"
             "(Dixon-Coles + 50,000 simulations)", color=NAVY, fontweight="bold", fontsize=12)
ax.xaxis.grid(True, color="#CBD5E1", lw=0.6)
ax.tick_params(colors=NAVY)
plt.tight_layout()
plt.savefig("viz_3_win_probs.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved → viz_3_win_probs.png")

### E4 · Bayesian Prior → Likelihood → Posterior

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
x = np.linspace(0, 5, 400)
panels = [
    ("Prior\n(before matches)",      sp_gamma(2,   scale=1/1.2), TEAL),
    ("Likelihood\n(from match data)", sp_gamma(6,   scale=1/0.5), GOLD),
    ("Posterior\n(updated belief)",   sp_gamma(8.5, scale=1/0.65), NAVY),
]
for ax, (title, dist, col) in zip(axes, panels):
    y = dist.pdf(x)
    ax.plot(x, y, color=col, lw=2.5)
    ax.fill_between(x, y, alpha=0.22, color=col)
    ax.set_title(title, color=NAVY, fontweight="bold", fontsize=11)
    ax.set_xlabel("Team attack strength", color=GRAY, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

# Arrows between panels
for i in range(2):
    fig.text(0.365 + i * 0.305, 0.52, "→",
             fontsize=26, color=TEAL, ha="center", va="center", fontweight="bold")

fig.suptitle("Bayesian Updating: Prior → Likelihood → Posterior",
             color=NAVY, fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("viz_4_bayesian_update.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved → viz_4_bayesian_update.png")

### E5 · Probabilistic Graphical Model (Plate Diagram)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 10); ax.set_ylim(0, 9.5); ax.axis("off")

def circ(cx, cy, label, sub="", col=TEAL):
    ax.add_patch(plt.Circle((cx, cy), 0.65, color=col, zorder=4, alpha=0.92))
    ax.text(cx, cy + (0.18 if sub else 0), label,
            ha="center", va="center", color=WHITE,
            fontsize=14, fontweight="bold", zorder=5)
    if sub:
        ax.text(cx, cy - 0.28, sub, ha="center", va="center",
                color=WHITE, fontsize=8.5, zorder=5)

def arw(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=NAVY, lw=2.2))

circ(5, 8.5, "μ",    "prior mean", col=GRAY)
circ(2, 6.2, "αᵢ",  "attack",     col=TEAL)
circ(8, 6.2, "δⱼ",  "defence",    col=TEAL)
circ(5, 4.0, "λᵢⱼ", "rate",       col=GOLD)
circ(5, 1.8, "Gᵢⱼ", "goals",      col=NAVY)

arw(5, 7.8, 2.6, 6.8)
arw(5, 7.8, 7.4, 6.8)
arw(2.65, 6.0, 4.35, 4.45)
arw(7.35, 6.0, 5.65, 4.45)
arw(5, 3.35, 5, 2.45)

# Plate (dashed box)
ax.add_patch(mpatches.FancyBboxPatch(
    (0.5, 0.9), 9.0, 4.3,
    boxstyle="round,pad=0.15", lw=2.2,
    edgecolor=TEAL, facecolor="none", linestyle="--", zorder=2))
ax.text(9.3, 1.1, "i, j\npairs", color=TEAL, fontsize=10, ha="right")

# Legend
for col, lbl in [(GRAY,"μ  global prior"), (TEAL,"αᵢ/δⱼ  team params"),
                 (GOLD,"λᵢⱼ  Poisson rate"), (NAVY,"Gᵢⱼ  observed goals")]:
    ax.add_patch(plt.Circle((-99,-99), 0.1, color=col, label=lbl))
ax.legend(loc="lower left", fontsize=9, framealpha=0.9, labelcolor=NAVY)

ax.set_title("Dixon-Coles as a Probabilistic Graphical Model",
             color=NAVY, fontweight="bold", fontsize=12, pad=12)
plt.tight_layout()
plt.savefig("viz_5_graphical_model.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved → viz_5_graphical_model.png")

### E6 · Calibration Plot (held-out predictions)

In [ ]:
# Use last 20% of matches as held-out set
cutoff  = int(len(df_wc) * 0.8)
df_held = df_wc.iloc[cutoff:].copy()

pred_hw, act_hw = [], []
for _, row in df_held.iterrows():
    lm, mu = match_rates(row["home_team"], row["away_team"])
    p_home = sum(
        poisson.pmf(hg, lm) * sum(poisson.pmf(ag, mu) for ag in range(hg))
        for hg in range(1, 9)
    )
    pred_hw.append(p_home)
    act_hw.append(1 if row["home_score"] > row["away_score"] else 0)

pa, ao = np.array(pred_hw), np.array(act_hw)
bidx   = np.digitize(pa, np.linspace(0, 1, 11)) - 1
bp, bo = [], []
for b in range(10):
    m = bidx == b
    if m.sum() > 0:
        bp.append(pa[m].mean()); bo.append(ao[m].mean())

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0,1], [0,1], "--", color=GRAY, lw=1.5, label="Perfect calibration")
ax.plot(bp, bo, "o-", color=TEAL, lw=2.2, ms=9,
        label="Dixon-Coles model", zorder=4)
ax.fill_between(bp, bo, bp, alpha=0.12, color=TEAL)
ax.set_xlabel("Predicted P(home win)", color=NAVY, fontweight="bold", fontsize=11)
ax.set_ylabel("Observed frequency",    color=NAVY, fontweight="bold", fontsize=11)
ax.set_title("Calibration Curve – Held-Out Matches",
             color=NAVY, fontweight="bold", fontsize=12)
ax.legend(framealpha=0, labelcolor=NAVY, fontsize=10)
ax.grid(True, color="#CBD5E1", lw=0.5)
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.tick_params(colors=NAVY)
plt.tight_layout()
plt.savefig("viz_6_calibration.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved → viz_6_calibration.png")

### Summary of saved files

| File | Description |
|------|-------------|
| `viz_1_poisson.png` | Poisson goal distribution (France vs England) |
| `viz_2_team_params.png` | Attack & defence ratings for all 20 teams |
| `viz_3_win_probs.png` | Win probabilities from 50,000 simulations |
| `viz_4_bayesian_update.png` | Prior → Likelihood → Posterior update |
| `viz_5_graphical_model.png` | Plate diagram of the DC graphical model |
| `viz_6_calibration.png` | Calibration on held-out matches |

**To download all files from Colab:**
```python
from google.colab import files
for f in ["viz_1_poisson.png","viz_2_team_params.png","viz_3_win_probs.png",
          "viz_4_bayesian_update.png","viz_5_graphical_model.png","viz_6_calibration.png"]:
    files.download(f)
```

In [ ]:
# ── Download all charts from Colab ────────────────────────────
from google.colab import files
for f in [
    "viz_1_poisson.png",
    "viz_2_team_params.png",
    "viz_3_win_probs.png",
    "viz_4_bayesian_update.png",
    "viz_5_graphical_model.png",
    "viz_6_calibration.png",
]:
    try:
        files.download(f)
        print(f"  Downloaded {f}")
    except Exception as e:
        print(f"  {f} not found yet – run the chart cells first")

## F · Optional — Full Bayesian Model with PyMC (MCMC)

This section uses **PyMC** to place proper priors over team strengths and sample from the posterior with MCMC. This gives genuine uncertainty quantification — not just point estimates.

**Runtime:** ~5–10 min on a Colab T4 GPU.
**Uncomment the code below to run.**

In [ ]:
# ── Uncomment everything below to run the PyMC model ─────────

# !pip install pymc arviz --quiet

# import pymc as pm
# import arviz as az

# teams_s  = sorted(set(df_wc["home_team"]) | set(df_wc["away_team"]))
# tidx_s   = {t: i for i, t in enumerate(teams_s)}
# ns       = len(teams_s)
# hi       = df_wc["home_team"].map(tidx_s).values
# ai       = df_wc["away_team"].map(tidx_s).values
# hg_obs   = df_wc["home_score"].values
# ag_obs   = df_wc["away_score"].values

# with pm.Model() as dc_bayes:
#     # Hierarchical priors on team strengths
#     mu_a   = pm.Normal("mu_a",  0, 1)
#     mu_d   = pm.Normal("mu_d",  0, 1)
#     sd_a   = pm.HalfNormal("sd_a", 1)
#     sd_d   = pm.HalfNormal("sd_d", 1)
#
#     att    = pm.Normal("att",  mu_a, sd_a, shape=ns)
#     dfc    = pm.Normal("dfc",  mu_d, sd_d, shape=ns)
#     home   = pm.Normal("home", 0.15, 0.2)   # home advantage prior
#
#     lam    = pm.math.exp(att[hi] - dfc[ai] + home)
#     mu2    = pm.math.exp(att[ai] - dfc[hi])
#
#     pm.Poisson("home_goals", lam,  observed=hg_obs)
#     pm.Poisson("away_goals", mu2,  observed=ag_obs)
#
#     trace = pm.sample(
#         1000, tune=500,
#         target_accept=0.9,
#         return_inferencedata=True,
#         progressbar=True
#     )

# # ── Posterior plots ───────────────────────────────────────────
# az.plot_posterior(trace, var_names=["home", "sd_a", "sd_d"])
# plt.savefig("viz_pymc_posteriors.png", dpi=150, bbox_inches="tight")
# plt.show()

# # ── Forest plot of team attack posteriors ─────────────────────
# az.plot_forest(trace, var_names=["att"],
#                coords={"att_dim_0": list(range(ns))},
#                combined=True, figsize=(8, 10))
# ax = plt.gca()
# ax.set_yticklabels(teams_s)
# ax.set_title("Posterior Attack Strengths (95% HDI)", fontweight="bold")
# plt.tight_layout()
# plt.savefig("viz_pymc_attack_forest.png", dpi=150, bbox_inches="tight")
# plt.show()

print("PyMC section is commented out. Uncomment to run MCMC.")